# DataFrames and joins

The same query as SQL and as DataFrame calls.

In [ ]:
import pathlib, sys, tempfile
sys.path.insert(0, "../datasets")  # docs/notebooks -> docs/datasets
import generate
import krishiv
from krishiv.sql import functions as F

data = pathlib.Path(tempfile.mkdtemp())
generate.generate(data, orders=20_000, customers=500)
spark = krishiv.session()
orders = spark.read.parquet(str(data / "orders.parquet"))
customers = spark.read.parquet(str(data / "customers.parquet"))
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

In [ ]:
by_region = (orders.join(customers, "customer_id")
    .groupBy("region").agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders"))
    .orderBy("region"))
by_region.show()

In [ ]:
sql = spark.sql("""SELECT region, sum(amount) AS revenue, count(*) AS orders
FROM orders JOIN customers USING (customer_id) GROUP BY region ORDER BY region""")
assert [(r[0], round(r[1], 2), r[2]) for r in sql.collect()] == [(r[0], round(r[1], 2), r[2]) for r in by_region.collect()]

In [ ]:
tbl = by_region.toArrow()  # a pyarrow.Table
assert tbl.num_rows == 4
tbl.schema